# From raw posts to the timeline

Exploratory notebook. The canonical English pipeline is `pipeline/` and the dashboard reads `data/01_filtered/posts.json` plus `data/02_peaks/peaks.json`. This notebook still uses its own helpers and writes inspection copies under `data/exports/`.

Functions are loaded from `pipeline/01_filter_posts.py` and `pipeline/02_detect_peaks.py`.

1. Look at the raw posts
2. Filter to project-related posts
3. Show the filtered posts
4. Split by channel
5. Language, as a draft
6. Cities through an extendable name list
7. Topic and word clusters, plus pie charts
8. Stance toward the project, separate from vendor tone. `-99` stays out of every mean
9. Timeline. Peak windows land in `data/exports/peak_windows/<project>/peak1`, `peak2`, …


In [ ]:
import importlib.util
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

def finde_datei(dateiname):
    hier = Path.cwd().resolve()
    kandidaten = []
    for ordner in (hier, hier.parent, hier.parent.parent):
        kandidaten.append(ordner)
        kandidaten.append(ordner / "Hackathon-KJRJ")
    gesehen = set()
    for ordner in kandidaten:
        if ordner in gesehen:
            continue
        gesehen.add(ordner)
        for kandidat in (ordner / "pipeline" / dateiname, ordner / dateiname):
            if kandidat.is_file():
                return kandidat
    raise FileNotFoundError(dateiname)


def modul_laden(dateiname, modulname):
    pfad = finde_datei(dateiname)
    spec = importlib.util.spec_from_file_location(modulname, pfad)
    modul = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(modul)
    projekt = pfad.parent.parent if pfad.parent.name == "pipeline" else pfad.parent
    return modul, projekt


filter_mod, SKRIPT_ORDNER = modul_laden("01_filter_posts.py", "filter_data")
try:
    peak_mod, _ = modul_laden("02_detect_peaks.py", "detect_peaks")
except Exception as fehler:
    peak_mod = None
    print("Peak-Modul nicht geladen, lokale Peak-Suche wird genutzt:", type(fehler).__name__)

DATEN_ORDNER = SKRIPT_ORDNER / "data"
# Raw file sits in the folder above the repo. See filter_mod.RAW_DIR.
ROH_PFAD = filter_mod.RAW_DIR / filter_mod.RAW_FILE_NAME
PEAK_ORDNER = DATEN_ORDNER / "exports" / "peak_windows"
print("Rohdaten:", ROH_PFAD)
print("Projekt:", SKRIPT_ORDNER)
print("Skripte:", SKRIPT_ORDNER / "pipeline")

SOCIAL_RELEVANT = frozenset({
    "YouTube", "Instagram", "Facebook", "Mastodon", "Flickr", "Reddit", "LinkedIn", "TikTok", "X",
})
PLATTFORM_DOMAINS = (
    ("YouTube", ("youtube.com", "youtu.be")),
    ("Instagram", ("instagram.com",)),
    ("Facebook", ("facebook.com",)),
    ("Reddit", ("reddit.com",)),
    ("LinkedIn", ("linkedin.com",)),
    ("TikTok", ("tiktok.com",)),
    ("Telegram", ("t.me", "telegram.me", "telegram.org")),
    ("Flickr", ("flickr.com",)),
    ("Pinterest", ("pinterest.com",)),
    ("Tumblr", ("tumblr.com",)),
    ("Bluesky", ("bsky.app",)),
    ("Vimeo", ("vimeo.com",)),
    ("X", ("twitter.com", "x.com")),
)
REGIONAL_HOST_MARKEN = (
    "meinbezirk", "anzeiger", "rundblick", "24hamburg", "bw24", "sauerlandkurier",
    "salzkammergut", "rheingau", "az-online", "krone.at", "kurier.at", "vol.at",
)

# Neue Städte: ein Eintrag mit Suchmustern genügt. Die Muster gelten für Titel und Text.
STAEDTE = {
    "Heilbronn": [r"heilbronn"],
    "Bonn": [r"\bbonn\b"],
    "Paris": [r"\bparis\b", r"cr[ée]teil", r"[îi]le-de-france", r"ile-de-france", r"val-de-marne"],
}

THEMEN = {
    "Kosten": ["kosten", "million", "euro", "budget", "teuer"],
    "Verkehr": ["öpnv", "oepnv", "nahverkehr", "stau", "bus", "pendler", "verkehr"],
    "Trasse": ["trasse", "strecke", "route", "stadtbild", "gondel"],
    "Politik": ["gemeinderat", "beschluss", "planung", "vorplanung", "stadtrat"],
    "Klima": ["klima", "nachhaltig", "umwelt", "lärm", "laerm"],
    "Campus und KI": ["ipai", "campus", "künstliche", "intelligenz"],
}

STOPPWOERTER = {
    "dass", "daß", "eine", "einer", "einem", "einen", "nicht", "auch", "sich", "sind",
    "wird", "wurde", "werden", "haben", "hat", "nach", "noch", "oder", "aber", "über",
    "unter", "zwischen", "sowie", "beim", "schon", "nur", "wie", "aus", "bei", "mit",
    "von", "für", "und", "der", "die", "das", "den", "dem", "des", "ein", "ist", "auf",
    "als", "zum", "zur", "vom", "im", "in", "am", "an", "zu", "es", "sie", "wir", "man",
    "les", "des", "une", "dans", "pour", "par", "sur", "avec", "plus", "que", "qui",
    "est", "sont", "pas", "aux", "cette", "dans", "leur", "leurs", "nous", "vous",
    "the", "and", "for", "with", "this", "that", "from", "https", "http", "www", "weiterlesen", "artikel", "diese", "dieser", "diesem", "durch", "allen",
}


def host_aus_quelle(quelle):
    host = str(quelle or "").strip().lower()
    return host[4:] if host.startswith("www.") else host


def host_ist(host, domain):
    return host == domain or host.endswith("." + domain)


def plattform_roh(quelle, medientyp):
    host = host_aus_quelle(quelle)
    for name, domains in PLATTFORM_DOMAINS:
        if any(host_ist(host, domain) for domain in domains):
            return name
    if medientyp == "Microblog":
        return "Mastodon"
    if medientyp in {"YouTube Video", "YouTube Comments"}:
        return "YouTube"
    return None


def plattform_bestimmen(quelle, medientyp):
    name = plattform_roh(quelle, medientyp)
    return name if name in SOCIAL_RELEVANT else None


def sprache_label(code):
    code = str(code or "").lower().split("-")[0]
    return {"de": "Deutsch", "fr": "Französisch", "en": "Englisch"}.get(code, code or "unbekannt")


def kanal_bestimmen(medientyp, plattform, host, titel, inhalt):
    if isinstance(plattform, str) and plattform:
        return "Social Media"
    if medientyp == "News" or any(marke in host for marke in REGIONAL_HOST_MARKEN):
        return "Nachrichten"
    return "Andere Webquellen"


def staedte_finden(text, regeln=None):
    regeln = STAEDTE if regeln is None else regeln
    text = str(text or "").lower()
    return [stadt for stadt, muster in regeln.items() if any(re.search(m, text) for m in muster)]


def thema_finden(text):
    text = str(text or "").lower()
    treffer = {name: sum(wort in text for wort in woerter) for name, woerter in THEMEN.items()}
    beste = max(treffer, key=treffer.get)
    return beste if treffer[beste] else "ohne Thema"


def tokens(text, limit=400):
    wort = re.findall(r"[a-zäöüßàâéèêëîïôùûç]{4,}", str(text or "").lower()[:limit])
    return [w for w in wort if w not in STOPPWOERTER]


def achsen_tausender(wert, _pos):
    return f"{int(wert):,}".replace(",", ".")


def kuchendiagramm(serie, titel):
    serie = serie[serie > 0].sort_values(ascending=False)
    if serie.empty:
        print(titel, ": keine Daten")
        return
    if len(serie) > 8:
        rest = serie.iloc[7:].sum()
        serie = serie.iloc[:7].copy()
        serie.loc["weitere"] = rest
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    ax.pie(serie.values, labels=[str(x) for x in serie.index], autopct="%1.0f%%", startangle=90)
    ax.set_title(titel)
    fig.tight_layout()
    plt.show()


def tagesreihe(rahmen):
    daten = rahmen.dropna(subset=["datum"]).copy()
    taeglich = daten.groupby("datum").size()
    index = pd.date_range(taeglich.index.min(), taeglich.index.max(), freq="D")
    return taeglich.reindex(index, fill_value=0)


PROJEKT_REGELN = {
    name: {
        "topic_terms": list(regeln["topic_terms"]),
        "context_terms": list(regeln["context_terms"]),
    }
    for name, regeln in filter_mod.PROJECT_BUZZWORDS.items()
}
PROJEKT_ORDNER_NAMEN = {
    "Heilbronn_Seilbahn": "Heilbronn",
    "Bonn_Seilbahn": "Bonn",
    "Paris_Cable_C1": "Paris",
    "IPAI_Campus": "IPAI",
}
PROJEKT_ORDNER = DATEN_ORDNER / "exports" / "by_project"


def projekte_aus_text(text, regeln=None):
    """Exactly one project, via assign_one_project in 01_filter_posts.py."""
    return filter_mod.assign_one_project(text, PROJEKT_REGELN if regeln is None else regeln)


def projektordner_name(schluessel):
    return PROJEKT_ORDNER_NAMEN.get(schluessel, schluessel)


def projekte_filtern(rahmen, regeln=None):
    """Beliebige Tabelle mit title und content auf die Projektregeln filtern."""
    rahmen = rahmen.copy()
    inhalt = rahmen["content"] if "content" in rahmen.columns else ""
    text = rahmen["title"].fillna("").astype(str) + "\n" + pd.Series(inhalt, index=rahmen.index).fillna("").astype(str)
    rahmen["projekte"] = text.map(lambda wert: projekte_aus_text(wert, regeln))
    return rahmen[rahmen["projekte"].map(bool)].copy()




## 1. Unstrukturierte Daten ansehen


In [ ]:
with open(ROH_PFAD, encoding="utf-8") as datei:
    roh_liste = filter_mod.extract_records(json.load(datei))

df_roh = pd.DataFrame(roh_liste)
df_roh["datum"] = pd.to_datetime(df_roh["published_at"], utc=True, errors="coerce").dt.tz_localize(None).dt.floor("D")
print(f"Rohbeiträge: {len(df_roh)}")
print("Zeitraum:", df_roh["datum"].min(), "bis", df_roh["datum"].max())
print("\nSprachen:")
print(df_roh["language_code"].fillna("leer").value_counts().head(12).to_string())
print("\nMedientypen:")
print(df_roh["media_type_name"].value_counts().to_string())
df_roh[["published_at", "language_code", "media_type_name", "source_name", "title"]].head(8)



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
df_roh["language_code"].fillna("leer").value_counts().head(8).plot(kind="barh", ax=axes[0], color="#3d7ec9")
axes[0].set_title("Rohdaten: Sprache")
df_roh["media_type_name"].value_counts().plot(kind="barh", ax=axes[1], color="#e07a3d")
axes[1].set_title("Rohdaten: Medientyp")
fig.tight_layout()
plt.show()

monat = df_roh.dropna(subset=["datum"]).groupby(df_roh["datum"].dt.to_period("M")).size()
fig, ax = plt.subplots(figsize=(11, 3.6))
monat.plot(ax=ax, color="#555555")
ax.set_title("Rohdaten: Beiträge pro Monat")
ax.set_ylabel("Beiträge")
fig.tight_layout()
plt.show()



## 2. Filtern

`evaluate_post` checks a valid date, then duplicates (title at least 90 percent and the same first 150 content characters, or the whole text at least 95 percent), then one project from `project_definitions.json`.


In [ ]:
gesehen = filter_mod.SeenPosts()
behalten = []
gruende = Counter()
for beitrag in roh_liste:
    gueltig, projekte, grund = filter_mod.evaluate_post(beitrag, gesehen)
    gruende[grund] += 1
    if gueltig:
        zeile = dict(beitrag)
        zeile["matched_projects"] = projekte
        behalten.append(zeile)

df = pd.DataFrame(behalten)
df["datum"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce").dt.tz_localize(None).dt.floor("D")
df["sprache"] = df["language_code"].map(sprache_label)
df["text"] = (df["title"].fillna("") + "\n" + df["content"].fillna("")).str.slice(0, 2500)
print(f"Behalten: {len(df)} von {len(df_roh)} ({len(df) / len(df_roh):.1%})")
print("\nAusschlussgründe:")
for grund, anzahl in gruende.most_common():
    if grund != "kept":
        print(f"  {anzahl:6d}  {grund}")
print("\nProjekte:")
print(df["matched_projects"].explode().value_counts().to_string())



## Projekte: Heilbronn, Bonn, Paris, IPAI

Ein Beitrag gehört zu genau einem Projekt: dem mit den meisten Muster-Treffern. Bei Gleichstand gilt die Reihenfolge in `PROJEKT_REGELN` (Heilbronn, Bonn, Paris, IPAI). Ein neues Projekt kommt als weiterer Eintrag dazu. `projekte_filtern` lässt sich auf jede Tabelle mit `title` und `content` anwenden.

Die Treffer liegen unter `data/exports/by_project/Heilbronn`, `data/exports/by_project/Bonn`, `data/exports/by_project/Paris` und `data/exports/by_project/IPAI`.

In [ ]:
# Neues Projekt auf weiteren Daten, zum Beispiel:
# PROJEKT_REGELN["Stuttgart_Seilbahn"] = {
#     "topic_terms": ["seilbahn"],
#     "context_terms": ["stuttgart"],
# }
# projekte_filtern(neue_tabelle)   # Tabelle braucht die Spalten title und content

df["projekte"] = (
    df["title"].fillna("").astype(str) + "\n" + df["content"].fillna("").astype(str)
).map(projekte_aus_text)
print(df["projekte"].explode().value_counts().to_string())

def beitragszeilen(rahmen):
    export = rahmen.copy()
    export["projekte_text"] = export["projekte"].map(lambda werte: ", ".join(werte) if isinstance(werte, list) else "")
    export["snippet"] = export["content"].fillna("").astype(str).str.slice(0, 400)
    spalten = [
        "mention_id", "published_at", "source_name", "title", "url",
        "language_code", "media_type_name", "projekte_text", "snippet",
    ]
    return export[[spalte for spalte in spalten if spalte in export.columns]]

if PROJEKT_ORDNER.exists():
    for alter in list(PROJEKT_ORDNER.iterdir()):
        if alter.is_dir():
            for datei in alter.glob("*"):
                datei.unlink()
            alter.rmdir()
PROJEKT_ORDNER.mkdir(parents=True, exist_ok=True)

uebersicht_projekte = []
for schluessel, regeln in PROJEKT_REGELN.items():
    name = projektordner_name(schluessel)
    teil = df[df["projekte"].map(lambda werte, ziel=schluessel: isinstance(werte, list) and ziel in werte)]
    ordner = PROJEKT_ORDNER / name
    ordner.mkdir(parents=True, exist_ok=True)
    beitragszeilen(teil).to_json(ordner / "posts.json", orient="records", force_ascii=False, indent=2)
    meta = {
        "projekt": name,
        "schluessel": schluessel,
        "beitraege": int(len(teil)),
        "topic_terms": regeln["topic_terms"],
        "context_terms": regeln["context_terms"],
    }
    (ordner / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    uebersicht_projekte.append(meta)
    print(f"{name}: {len(teil)} Beiträge -> {ordner.name}/")

(PROJEKT_ORDNER / "index.json").write_text(
    json.dumps(uebersicht_projekte, ensure_ascii=False, indent=2), encoding="utf-8"
)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
pd.Series({eintrag["projekt"]: eintrag["beitraege"] for eintrag in uebersicht_projekte}).plot(kind="barh", ax=ax, color="#3d7ec9")
ax.set_title("Gefilterte Daten nach Projekt")
ax.set_xlabel("Beiträge")
fig.tight_layout()
plt.show()
print("Abgelegt unter:", PROJEKT_ORDNER)

## 3. Gefilterte Daten erkunden


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
pd.Series({"Rohdaten": len(df_roh), "Gefiltert": len(df)}).plot(kind="bar", ax=axes[0], color=["#bbbbbb", "#e07a3d"])
axes[0].set_title("Vorher / nachher")
axes[0].tick_params(axis="x", rotation=0)
df["matched_projects"].explode().value_counts().plot(kind="barh", ax=axes[1], color="#3d7ec9")
axes[1].set_title("Gefilterte Daten: Projekte")
fig.tight_layout()
plt.show()

taeglich = tagesreihe(df)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(taeglich.index, taeglich.values, color="#333333", linewidth=1.2)
ax.set_title("Gefilterte Daten: Beiträge pro Tag")
ax.set_ylabel("Beiträge")
fig.tight_layout()
plt.show()



## 4. Aufteilung nach Medium

Erst auf den gefilterten Beiträgen. Social Media enthält nur die relevanten Plattformen. Klassische Medien und regionale Nachrichtenquellen heißen beide Nachrichten. Urlaub ist keine eigene Kategorie.

In [ ]:
df["plattform"] = [plattform_bestimmen(q, m) for q, m in zip(df["source_name"], df["media_type_name"])]
df["kanal"] = [
    kanal_bestimmen(m, p, host_aus_quelle(q), t, c)
    for m, p, q, t, c in zip(df["media_type_name"], df["plattform"], df["source_name"], df["title"], df["content"])
]
print("Kanal:")
print(df["kanal"].value_counts().to_string())
print("\nMedientyp:")
print(df["media_type_name"].value_counts().to_string())
print("\nRelevante Social-Media-Quellen:")
print(df.loc[df["kanal"].eq("Social Media"), "plattform"].value_counts().to_string())
kuchendiagramm(df["kanal"].value_counts(), "Gefilterte Daten: Kanal")
kuchendiagramm(df["media_type_name"].value_counts(), "Gefilterte Daten: Medientyp")



## 5. Aufteilung nach Sprache

Entwurf aus `language_code`. Regionale Codes wie `de-DE` werden auf `Deutsch` zusammengezogen.


In [ ]:
print("Entwurf Sprache (aus language_code, auf die Hauptsprache gekürzt):")
print(df["sprache"].value_counts().to_string())
kuchendiagramm(df["sprache"].value_counts(), "Gefilterte Daten: Sprache")
sprachen_zeit = (
    df.dropna(subset=["datum"])
    .groupby([df["datum"].dt.to_period("M"), "sprache"])
    .size()
    .unstack(fill_value=0)
)
fig, ax = plt.subplots(figsize=(11, 3.8))
sprachen_zeit.plot(ax=ax)
ax.set_title("Entwurf: Beiträge pro Monat nach Sprache")
ax.set_ylabel("Beiträge")
fig.tight_layout()
plt.show()



## 6. Aufteilung nach Stadt

Neue Städte kommen als weiterer Eintrag in `STAEDTE` dazu, jeweils mit einem oder mehreren Suchmustern. Die Liste hängt nicht an einem festen Dateinamen.


In [ ]:
df["staedte"] = df["text"].map(staedte_finden)
df["stadt"] = df["staedte"].map(lambda namen: namen[0] if len(namen) == 1 else ("mehrere Städte" if namen else "ohne Stadt"))
print("Eine Zeile kann mehrere Städte enthalten. Für das Diagramm zählt die erste eindeutige Stadt.")
print(df["stadt"].value_counts().to_string())
print("\nMehrfachnennung (eine Zeile zählt in jeder gefundenen Stadt):")
print(df["staedte"].explode().dropna().value_counts().to_string())
kuchendiagramm(df["stadt"].value_counts(), "Gefilterte Daten: Stadt")



## 7. Cluster und Kuchendiagramme

Die Themencluster nutzen feste Begriffslisten. Die Wortcluster sind ein Entwurf: jeder Beitrag wird über seine Wörter einer von fünf Gruppen zugeordnet.


In [ ]:
df["thema"] = df["text"].map(thema_finden)
print("Themencluster über feste Begriffslisten:")
print(df["thema"].value_counts().to_string())
kuchendiagramm(df["thema"].value_counts(), "Themencluster")

wort_zaehler = Counter()
for text in df["text"]:
    wort_zaehler.update(set(tokens(text)))
vokabular = [wort for wort, _ in wort_zaehler.most_common(80)]
index = {wort: i for i, wort in enumerate(vokabular)}
matrix = np.zeros((len(df), len(vokabular)), dtype=float)
for zeile, text in enumerate(df["text"]):
    for wort in set(tokens(text)):
        if wort in index:
            matrix[zeile, index[wort]] = 1.0
norm = np.linalg.norm(matrix, axis=1, keepdims=True)
matrix = matrix / np.clip(norm, 1e-9, None)

k = 5
zufall = np.random.default_rng(7)
zentren = matrix[zufall.choice(len(matrix), size=k, replace=False)]
labels = np.zeros(len(matrix), dtype=int)
for _ in range(12):
    abstand = ((matrix[:, None, :] - zentren[None, :, :]) ** 2).sum(axis=2)
    labels = abstand.argmin(axis=1)
    for gruppe in range(k):
        if np.any(labels == gruppe):
            zentren[gruppe] = matrix[labels == gruppe].mean(axis=0)

df["wortcluster"] = [f"Cluster {n + 1}" for n in labels]
print("\nAutomatische Wortcluster (Entwurf, häufigste Begriffe):")
for gruppe in range(k):
    gewichte = zentren[gruppe]
    begriffe = [vokabular[i] for i in np.argsort(gewichte)[-6:][::-1]]
    print(f"  Cluster {gruppe + 1} ({int((labels == gruppe).sum())} Beiträge): {', '.join(begriffe)}")
kuchendiagramm(df["wortcluster"].value_counts(), "Wortcluster")



## 8. Haltung zum Projekt, getrennt von der Vendor-Tonlage

Zwei Spalten, zwei Quellen.

**Vendor-Tonlage** ist das mitgelieferte Feld `sentiment`: `1` positiv, `0` neutral, `-1` negativ. `-99` ist keine Angabe. `mittelwert` streicht `-99` und leere Werte, bevor es mittelt. Dieselbe Streichung macht `pipeline/02_detect_peaks.py` bei jedem Tonlage-Mittelwert.

**Haltung zum Projekt** liest nur den Text, über die Muster in dieser Zelle. Die Vendor-Zahl geht dort nicht ein. Treffen dafür und dagegen zusammen, oder ein Abwägungsmuster, lautet die Haltung `abwägend`. Ohne Treffer bleibt `keine erkennbare Haltung`.

Die Muster hängen am Projektwort: Seilbahn, Téléphérique, Câble C1, Gondelbahn. „Vorplanung für die Seilbahn“ und „soll den Verkehr entlasten“ zählen nicht als Haltung.


In [ ]:
FEHLWERT = -99
TONLAGE_LABEL = {1.0: "positiv", 0.0: "neutral", -1.0: "negativ"}
HALTUNG_REIHENFOLGE = ["dafür", "dagegen", "abwägend", "keine erkennbare Haltung"]
OBJEKT = r"(?:seilbahn\w*|t[ée]l[ée]ph[ée]rique\w*|c[âa]ble\s*c\s*1|gondelbahn\w*|stadtseilbahn\w*)"


def muster(schablone):
    return schablone.replace("{objekt}", OBJEKT).replace("{nah}", r".{0,60}")


HALTUNG_DAFUER = [
    muster(r"ja zur {objekt}"),
    muster(r"gr[üu]nes licht f[üu]r (?:die |eine |konkrete )?(?:vorplanung|{objekt})"),
    muster(r"bef[üu]rwort\w*(?:\s+\w+){0,4}\s+{objekt}"),
    muster(r"begr[üu][sß]\w*(?:\s+\w+){0,3}\s+(?:die |das )?{objekt}"),
    muster(r"(?:sieht|sehen)\s+gro[sß]e chancen{nah}{objekt}|{objekt}{nah}(?:sieht|sehen)\s+gro[sß]e chancen"),
    muster(r"chancen steigen{nah}{objekt}|{objekt}{nah}chancen steigen"),
    r"positiv\w*\s+machbarkeitsstudie",
]
HALTUNG_DAGEGEN = [
    r"luftnummer",
    muster(r"prestigeprojekt\w*{nah}(?:geld|k[üu]rzung)|k[üu]rzung{nah}prestigeprojekt"),
    muster(r"\bnein\b.{0,30}zur {objekt}|sagen\s+[\"“„]?nein.{0,40}{objekt}"),
    muster(r"\bkeine {objekt} (?:über|ueber|in|nach|durch|für|fuer)\b"),
    muster(r"gegen die (?:geplante )?{objekt}"),
    muster(r"{objekt}{nah}abgelehnt|abgelehnt{nah}{objekt}"),
    muster(r"{objekt}{nah}kritisch|kritisch{nah}{objekt}"),
    muster(r"unsinnig\w*{nah}{objekt}|{objekt}{nah}unsinn"),
    muster(r"geldverschwend\w*{nah}{objekt}|{objekt}{nah}geldverschwend"),
    muster(r"s'oppose\w*{nah}{objekt}|{objekt}{nah}s'oppose\w*"),
]
HALTUNG_ABWAEGEND = [
    muster(r"pro und contra{nah}{objekt}|{objekt}{nah}pro und contra"),
    muster(r"vor- und nachteile{nah}{objekt}|{objekt}{nah}vor- und nachteile"),
    muster(r"chancen und risiken{nah}{objekt}|{objekt}{nah}chancen und risiken"),
    muster(r"gro[sß]e chancen{nah}risiken|warnt vor risiken{nah}{objekt}"),
]


def mittelwert(werte):
    """Mittelwert ohne -99 und ohne leere Angaben."""
    zahlen = pd.to_numeric(pd.Series(werte), errors="coerce")
    gueltig = zahlen[zahlen.ne(FEHLWERT) & zahlen.notna()]
    if gueltig.empty:
        return float("nan")
    return float(gueltig.mean())


def vendor_tonlage(wert):
    zahl = pd.to_numeric(wert, errors="coerce")
    if pd.isna(zahl) or float(zahl) == FEHLWERT:
        return "keine Angabe"
    return TONLAGE_LABEL.get(float(zahl), "keine Angabe")


def haltung_aus_text(text):
    text = str(text or "").lower()
    dafuer = any(re.search(eintrag, text) for eintrag in HALTUNG_DAFUER)
    dagegen = any(re.search(eintrag, text) for eintrag in HALTUNG_DAGEGEN)
    abwaegend = any(re.search(eintrag, text) for eintrag in HALTUNG_ABWAEGEND)
    if abwaegend or (dafuer and dagegen):
        return "abwägend"
    if dafuer:
        return "dafür"
    if dagegen:
        return "dagegen"
    return "keine erkennbare Haltung"


df["vendor_tonlage"] = df["sentiment"].map(vendor_tonlage)
df["haltung"] = df["text"].map(haltung_aus_text)
df["projektname"] = df["matched_projects"].map(
    lambda werte: projektordner_name(werte[0]) if isinstance(werte, list) and werte else "ohne Projekt"
)

roh_tonlage = pd.to_numeric(df["sentiment"], errors="coerce")
ausgelassen = int((roh_tonlage.eq(FEHLWERT) | roh_tonlage.isna()).sum())
print(
    f"Vendor-Tonlage, Mittelwert: {mittelwert(roh_tonlage):.2f} "
    f"aus {int((roh_tonlage.ne(FEHLWERT) & roh_tonlage.notna()).sum())} Angaben, "
    f"{ausgelassen} mal -99 oder leer ausgelassen"
)
print("\nVendor-Tonlage:")
print(df["vendor_tonlage"].value_counts().reindex(["positiv", "neutral", "negativ", "keine Angabe"]).to_string())
print("\nHaltung zum Projekt:")
print(df["haltung"].value_counts().reindex(HALTUNG_REIHENFOLGE).to_string())
print("\nHaltung neben Vendor-Tonlage (Anzahl Beiträge):")
print(pd.crosstab(df["haltung"], df["vendor_tonlage"]).reindex(index=HALTUNG_REIHENFOLGE).to_string())

tonlage_je_projekt = []
for name, gruppe in df.groupby("projektname"):
    zahlen = pd.to_numeric(gruppe["sentiment"], errors="coerce")
    tonlage_je_projekt.append({
        "Projekt": name,
        "Beiträge": int(len(gruppe)),
        "Mittelwert": round(mittelwert(zahlen), 2),
        "Angaben im Mittelwert": int((zahlen.ne(FEHLWERT) & zahlen.notna()).sum()),
        "-99 oder leer": int((zahlen.eq(FEHLWERT) | zahlen.isna()).sum()),
    })
print("\nVendor-Tonlage je Projekt, Mittelwert ohne -99:")
print(pd.DataFrame(tonlage_je_projekt).set_index("Projekt").to_string())
print("\nHaltung je Projekt:")
print(pd.crosstab(df["projektname"], df["haltung"]).reindex(columns=HALTUNG_REIHENFOLGE).to_string())

for code in ("dafür", "dagegen", "abwägend"):
    print(f"\nBeispiele, {code}:")
    titel = df.loc[df["haltung"].eq(code), "title"].dropna().astype(str).head(3)
    if titel.empty:
        print("  keine")
    for zeile in titel:
        print(" ", zeile[:110])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
(
    df["haltung"].value_counts().reindex(HALTUNG_REIHENFOLGE).fillna(0)
    .plot(kind="barh", ax=axes[0], color="#3d7ec9")
)
axes[0].set_title("Haltung zum Projekt")
axes[0].set_xlabel("Beiträge")
(
    df["vendor_tonlage"].value_counts().reindex(["positiv", "neutral", "negativ", "keine Angabe"]).fillna(0)
    .plot(kind="barh", ax=axes[1], color="#e07a3d")
)
axes[1].set_title("Vendor-Tonlage")
axes[1].set_xlabel("Beiträge")
fig.tight_layout()
plt.show()


## 9. Zeitstrahl mit Peaks

Zuerst der gesamte gefilterte Verlauf. Die Peaks kommen danach je Projekt aus `pipeline/02_detect_peaks.py`: `scipy.signal.find_peaks` mit Prominenz und mindestens sieben Tagen Abstand. Die Zahl an einem Peak ist der Mittelwert der Vendor-Tonlage. `-99` wird vorher geleert und geht nicht ein. Die Haltung zum Projekt aus Abschnitt 8 wird hier nicht gemittelt.

Die Beiträge im Fenster von drei Tagen davor bis zwei Tage danach liegen unter `data/exports/peak_windows/<Projekt>/peak1`, `peak2` und so weiter.

In [ ]:
if peak_mod is None:
    raise RuntimeError("2_detect_peaks.py ist nicht geladen. scipy muss installiert sein.")

reihe = tagesreihe(df)
fig, ax = plt.subplots(figsize=(11.5, 4.2))
ax.plot(reihe.index, reihe.values, color="#333333", linewidth=1.3)
ax.fill_between(reihe.index, reihe.values, color="#333333", alpha=0.08)
ax.set_title("Gefilterte Daten: Zeitstrahl")
ax.set_ylabel("Beiträge pro Tag")
ax.set_xlabel("Datum")
fig.tight_layout()
plt.show()

arbeit = df.dropna(subset=["datum"]).copy()
arbeit["date"] = arbeit["datum"]
arbeit["sentiment"] = pd.to_numeric(arbeit["sentiment"], errors="coerce")
arbeit.loc[arbeit["sentiment"].eq(FEHLWERT), "sentiment"] = np.nan
arbeit["project"] = arbeit["matched_projects"].map(
    lambda werte: werte[0] if isinstance(werte, list) and werte else None
)
arbeit = arbeit.dropna(subset=["project"])


def ordner_leeren(wurzel):
    if not wurzel.exists():
        return
    for pfad in sorted(wurzel.rglob("*"), reverse=True):
        if pfad.is_file() or pfad.is_symlink():
            pfad.unlink()
        elif pfad.is_dir():
            pfad.rmdir()


ordner_leeren(PEAK_ORDNER)
PEAK_ORDNER.mkdir(parents=True, exist_ok=True)

farben = {
    "Heilbronn_Seilbahn": "#e07a3d",
    "Bonn_Seilbahn": "#3d7ec9",
    "Paris_Cable_C1": "#2f9e8f",
    "IPAI_Campus": "#7a4ea3",
}
reihenfolge = [name for name in PROJEKT_REGELN if name in set(arbeit["project"])]
uebersicht = []
trendbrueche = []
fig, axes = plt.subplots(len(reihenfolge), 1, figsize=(11.5, 2.5 * max(len(reihenfolge), 1)), sharex=True)
if len(reihenfolge) == 1:
    axes = [axes]

for ax, projekt in zip(axes, reihenfolge):
    gruppe = arbeit[arbeit["project"] == projekt]
    ergebnis = peak_mod.analyze_project_timeseries(gruppe, projekt)
    tages = tagesreihe(gruppe)
    ax.plot(tages.index, tages.values, color=farben.get(projekt, "#333333"), linewidth=1.2)
    name = projektordner_name(projekt)
    print(f"\n{name}: {ergebnis['total_posts']} Beiträge, {len(ergebnis['peaks'])} Peaks")
    for cp in ergebnis["change_points"]:
        cp = dict(cp)
        cp["projekt"] = name
        trendbrueche.append(cp)
        richtung = "Anstieg" if cp["avg_posts_after"] > cp["avg_posts_before"] else "Rückgang"
        print(
            f"  Trendbruch {cp['date']} ({richtung}): "
            f"{cp['avg_posts_before']} -> {cp['avg_posts_after']} Beiträge/Tag, "
            f"Vendor-Tonlage {cp['avg_sentiment_before']} -> {cp['avg_sentiment_after']}"
        )
    for nummer, peak in enumerate(ergebnis["peaks"], start=1):
        tag = pd.Timestamp(peak["date"])
        start = pd.Timestamp(peak["crawler_window"]["start"])
        ende = pd.Timestamp(peak["crawler_window"]["end"])
        ax.scatter([tag], [peak["post_count"]], color="#111111", s=28, zorder=3)
        ax.annotate(f"P{nummer}", (tag, peak["post_count"]), textcoords="offset points", xytext=(4, 6), fontsize=8)
        fenster = gruppe[gruppe["datum"].between(start, ende)].copy()
        ordner = PEAK_ORDNER / name / f"peak{nummer}"
        ordner.mkdir(parents=True, exist_ok=True)
        begriffe = Counter()
        for text in fenster["text"]:
            begriffe.update(tokens(text, limit=800))
        top = begriffe.most_common(15)
        meta = {
            "projekt": name,
            "peak": nummer,
            "datum": peak["date"],
            "beitraege_am_tag": int(peak["post_count"]),
            "prominenz": peak["prominence"],
            "vendor_tonlage": peak["avg_sentiment"],
            "beitraege_im_fenster": int(len(fenster)),
            "fenster": {"start": peak["crawler_window"]["start"], "ende": peak["crawler_window"]["end"]},
            "haeufigste_begriffe": [{"begriff": wort, "anzahl": anzahl} for wort, anzahl in top],
        }
        export = fenster.assign(
            snippet=fenster["content"].fillna("").astype(str).str.slice(0, 400),
            projekte=name,
        )[
            ["mention_id", "published_at", "source_name", "title", "url", "language_code", "media_type_name", "projekte", "snippet"]
        ]
        export.to_json(ordner / "posts.json", orient="records", force_ascii=False, indent=2)
        (ordner / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
        uebersicht.append(meta)
        print(
            f"  Peak {nummer} — {meta['datum']} — {meta['beitraege_am_tag']} Beiträge, "
            f"Vendor-Tonlage {meta['vendor_tonlage']}"
        )
        print("  Häufigste Begriffe:", ", ".join(f"{wort} ({anzahl})" for wort, anzahl in top[:8]))
    ax.set_ylabel(name)

axes[-1].set_xlabel("Datum")
fig.suptitle("Peaks je Projekt", y=1.01)
fig.tight_layout()
plt.show()
(PEAK_ORDNER / "index.json").write_text(
    json.dumps({"peaks": uebersicht, "trendbrueche": trendbrueche}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nAbgelegt unter:", PEAK_ORDNER)

